# Real vs. Euclidean distance — mean golden ratio

Computes the ratio of real road-network distance to Euclidean distance for every Amazon delivery route (the full route, depot through all stops and
back), and reports a **single mean golden ratio** over all routes across all five metropolitan areas.

**Before running:**
1. Download the dataset — see `docs/DATA.md`.
2. Start one OSRM server per metro — see `docs/OSRM.md`.
3. Set `data_dir` in the loader cell below.

Run all cells top to bottom. The final cell prints the headline number.


## Step 1 — Load the data

Set `data_dir` to your local `model_build_inputs/` folder (see `docs/DATA.md`).
This cell builds `stops_df`, `routes_df`, `packages_df`, and `actual_sequences`,
and tags every stop with its metro via the station-code prefix. No edits needed
beyond the path.


In [10]:
# ============================================================
# STEP 1: DATA LOADER
# Defines: stops_df, routes_df, actual_sequences
# ============================================================
import os, json
import numpy as np
import pandas as pd

# --- Paths ---
data_dir = "./almrrc2021-data-training/model_build_inputs"  # <-- change to your local path
route_data_file       = os.path.join(data_dir, "route_data.json")
package_data_file     = os.path.join(data_dir, "package_data.json")
actual_sequences_file = os.path.join(data_dir, "actual_sequences.json")

# --- Load route + stop data ---
with open(route_data_file, "r") as f:
    route_data = json.load(f)

routes_list, stops_list = [], []
for route_id, rdata in route_data.items():
    routes_list.append({
        "route_id": route_id,
        "station_code": rdata["station_code"],
        "date": rdata["date_YYYY_MM_DD"],
        "departure_time_utc": rdata["departure_time_utc"],
        "executor_capacity_cm3": rdata["executor_capacity_cm3"],
        "route_score": rdata["route_score"],
    })
    for stop_id, sdata in rdata["stops"].items():
        stops_list.append({
            "route_id": route_id,
            "stop_id": stop_id,
            "lat": sdata["lat"],
            "lng": sdata["lng"],
            "type": sdata["type"],
            "zone_id": sdata["zone_id"],
        })

routes_df = pd.DataFrame(routes_list)
stops_df  = pd.DataFrame(stops_list)
print(f"Routes: {routes_df.shape}, Stops: {stops_df.shape}")

# --- Load package data (optional; kept for parity with your workflow) ---
with open(package_data_file, "r") as f:
    package_data = json.load(f)
packages_list = []
for route_id, stops in package_data.items():
    for stop_id, pkgs in stops.items():
        for pkg_id, pdata in pkgs.items():
            dim = pdata.get("dimensions", {})
            tw  = pdata.get("time_window", {})
            packages_list.append({
                "route_id": route_id, "stop_id": stop_id, "package_id": pkg_id,
                "scan_status": pdata.get("scan_status"),
                "start_time_utc": tw.get("start_time_utc"),
                "end_time_utc": tw.get("end_time_utc"),
                "planned_service_time_seconds": pdata.get("planned_service_time_seconds"),
                "depth_cm": dim.get("depth_cm"), "height_cm": dim.get("height_cm"),
                "width_cm": dim.get("width_cm"),
            })
packages_df = pd.DataFrame(packages_list)
print(f"Packages: {packages_df.shape}")

# --- Load actual visit sequences ---
# ALMRRC stores order as actual_sequences[route_id]["actual"] = {stop_id: order_index}
with open(actual_sequences_file, "r") as f:
    actual_sequences = json.load(f)
print(f"Sequences: {len(actual_sequences)} routes")

# --- Metro assignment ---
# ALMRRC station codes carry the metro in their prefix; this is exact.
_STATION_PREFIX_TO_METRO = {
    "DSE": "Seattle", "DLA": "Los Angeles", "DAU": "Austin",
    "DCH": "Chicago", "DBO": "Boston",
}
# Map each route -> metro via its station_code prefix
_route_to_metro = {
    r.route_id: _STATION_PREFIX_TO_METRO.get(str(r.station_code)[:3])
    for r in routes_df.itertuples()
}
stops_df["area"] = stops_df["route_id"].map(_route_to_metro)
print("Metros:", stops_df["area"].value_counts().to_dict())


Routes: (6112, 6), Stops: (904527, 6)
Packages: (1457175, 10)
Sequences: 6112 routes
Metros: {'Los Angeles': 414440, 'Chicago': 162410, 'Seattle': 155781, 'Boston': 140622, 'Austin': 31274}


## Step 2 — Configuration

Map each metro to the port its OSRM server is running on. If you run one server at
a time on port 5000, set every value to 5000 and the script will process whichever
metro is currently served (others will be skipped with a message).


In [11]:
OSRM_PORTS = {
    "Seattle": 5001,
    "Los Angeles": 5002,
    "Chicago": 5003,
    "Boston": 5004,
    "Austin": 5005,
}

PATH_BATCH = 90  # coords per OSRM /route call (stays under URL limits)


## Step 3 — Distance helpers

In [12]:
import numpy as np
import pandas as pd
import requests

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlam/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def eucl_km(df):
    """Straight-line length along ordered stops."""
    if len(df) < 2:
        return 0.0
    lat, lng = df["lat"].to_numpy(), df["lng"].to_numpy()
    return haversine_km(lat[:-1], lng[:-1], lat[1:], lng[1:]).sum()

def osrm_km(df, base_url):
    """Real road length along ordered stops, via a local OSRM server.
    Long paths are chunked with a one-point overlap so no leg is dropped."""
    pts = list(zip(df["lat"].to_numpy(), df["lng"].to_numpy()))
    if len(pts) < 2:
        return 0.0
    total, i = 0.0, 0
    while i < len(pts) - 1:
        chunk = pts[i:i + PATH_BATCH]
        if len(chunk) < 2:
            break
        coord_str = ";".join(f"{lng},{lat}" for lat, lng in chunk)
        r = requests.get(f"{base_url}/route/v1/driving/{coord_str}",
                         params={"overview": "false", "continue_straight": "true"},
                         timeout=60)
        r.raise_for_status()
        d = r.json()
        if d.get("code") != "Ok":
            raise RuntimeError(d.get("code"))
        total += d["routes"][0]["distance"] / 1000
        i += PATH_BATCH - 1
    return total

def closed_tour(depot, stops):
    """The full route: depot -> all stops -> depot."""
    if len(depot):
        return pd.concat([depot.iloc[[0]], stops, depot.iloc[[0]]], ignore_index=True)
    return pd.concat([stops, stops.iloc[[0]]], ignore_index=True)

def server_up(base_url):
    try:
        requests.get(f"{base_url}/route/v1/driving/-122.33,47.60;-122.34,47.61",
                     params={"overview": "false"}, timeout=5)
        return True
    except Exception:
        return False


## Step 4 — Compute the golden ratio for every route

For each metro whose OSRM server is up, this walks its routes, builds the ordered
stop sequence from `actual_sequences`, and computes Euclidean and real length for the full route
(depot through all stops and back). Progress prints every 50 routes as **it might take several minutes**.


In [13]:
import time

records = []
for METRO, port in OSRM_PORTS.items():
    base_url = f"http://localhost:{port}"
    if not server_up(base_url):
        print(f"SKIP {METRO}: no OSRM server on {base_url}")
        continue

    metro_routes = [rid for rid in route_area[route_area == METRO].index
                    if rid in actual_sequences]
    print(f"{METRO}: {len(metro_routes)} routes (port {port})")

    t0, done = time.time(), 0
    for rid in metro_routes:
        try:
            order_map = actual_sequences[rid]["actual"]
            rs = stops_df[stops_df["route_id"] == rid].copy()
            rs["visit_order"] = rs["stop_id"].map(order_map)
            rs = (rs.dropna(subset=["visit_order"])
                    .sort_values("visit_order").reset_index(drop=True))
            if len(rs) < 2:
                continue

            is_depot = rs["type"].str.lower().eq("station")
            depot = rs[is_depot]
            deliv = rs[~is_depot].reset_index(drop=True)
            if len(deliv) < 2:
                continue

            # Full route: depot -> all stops -> depot
            tour = closed_tour(depot, deliv)
            eucl = eucl_km(tour)
            real = osrm_km(tour, base_url)
            if eucl <= 0:
                continue

            records.append({
                "route_id": rid, "metro": METRO, "n_stops": len(deliv),
                "golden_ratio": real / eucl,
            })
        except Exception as e:
            records.append({"route_id": rid, "metro": METRO, "n_stops": np.nan,
                            "golden_ratio": np.nan, "error": str(e)})
        done += 1
        if done % 50 == 0:
            print(f"   {done}/{len(metro_routes)}  ({time.time()-t0:.0f}s)")

results = pd.DataFrame(records)
results.to_csv("route_golden_ratios.csv", index=False)
ok = int(results["golden_ratio"].notna().sum())
print(f"\nComputed {ok} routes ({int(results['golden_ratio'].isna().sum())} failed). "
      f"Saved route_golden_ratios.csv")


Seattle: 1079 routes (port 5001)
   50/1079  (6s)
   100/1079  (12s)
   150/1079  (17s)
   200/1079  (23s)
   250/1079  (29s)
   300/1079  (35s)
   350/1079  (40s)
   400/1079  (45s)
   450/1079  (50s)
   500/1079  (56s)
   550/1079  (61s)
   600/1079  (66s)
   650/1079  (72s)
   700/1079  (78s)
   750/1079  (84s)
   800/1079  (89s)
   850/1079  (95s)
   900/1079  (100s)
   950/1079  (105s)
   1000/1079  (111s)
   1050/1079  (117s)
Los Angeles: 2888 routes (port 5002)
   50/2888  (6s)
   100/2888  (11s)
   150/2888  (17s)
   200/2888  (22s)
   250/2888  (28s)
   300/2888  (34s)
   350/2888  (40s)
   400/2888  (46s)
   450/2888  (51s)
   500/2888  (57s)
   550/2888  (62s)
   600/2888  (68s)
   650/2888  (74s)
   700/2888  (80s)
   750/2888  (86s)
   800/2888  (92s)
   850/2888  (98s)
   900/2888  (104s)
   950/2888  (109s)
   1000/2888  (116s)
   1050/2888  (121s)
   1100/2888  (127s)
   1150/2888  (133s)
   1200/2888  (139s)
   1250/2888  (144s)
   1300/2888  (150s)
   1350/2888  (156s

## Step 5 — The ratio!

The mean golden ratio over all routes, all metros.

In [17]:
d = results[["golden_ratio", "metro"]].dropna()
d = d[(d["golden_ratio"] > 1.0) & (d["golden_ratio"] < 3.0)]   # physical range

# Pool all routes: each metro contributes in proportion to its number of routes.
vals = d["golden_ratio"].to_numpy()
mean_golden_ratio = vals.mean()

print(f"Routes included: {len(d)}")
print(f"Routes per metro: {d['metro'].value_counts().to_dict()}")
print()
print(f"  MEAN GOLDEN RATIO = {mean_golden_ratio:.2f}")
print()
print(f"  ≈ e^{np.log(mean_golden_ratio):.2f}")
print()


Routes included: 6103
Routes per metro: {'Los Angeles': 2880, 'Seattle': 1079, 'Chicago': 1002, 'Boston': 928, 'Austin': 214}

  MEAN GOLDEN RATIO = 1.61

  ≈ e^0.48

